# Phase 4 — Evaluation & Error Analysis (Catanduanes Bikol NMT)

This measures how much the LoRA fine-tuning helped, on the **held-out test set** (touched here
for the first time). It produces:
- **BLEU, chrF++, TER** for the **base** model vs your **fine-tuned** model (criteria D1, D2)
- A **comparison table** (before vs after)
- An **error-analysis table** of the worst 40 test sentences to categorize (criterion D3)

**Decoding = greedy (num_beams=1)** — the same setting your API ships, so the numbers match the demo.
Set **Runtime → T4 GPU**, then Run all.

## 1. Install dependencies

In [ ]:
!pip -q install "transformers==4.46.3" "tokenizers==0.20.3" "peft==0.13.2" \
    "accelerate==1.1.1" "sacrebleu==2.4.3" sentencepiece
import torch; print("CUDA:", torch.cuda.is_available(), "| done")

## 2. Get code + adapter + test set from GitHub

In [ ]:
!rm -rf bikol-nmt && git clone -q https://github.com/LenardBenavidez/bikol-nmt.git
import os, pandas as pd
os.chdir("bikol-nmt")
test_df = pd.read_csv("data/processed_sentences/test.csv").dropna()
SRC_COL, TGT_COL = "Bicol", "English"
print("test pairs:", len(test_df))
test_df.head()

## 3. Load both models
`base` = un-tuned NLLB-200 (the baseline to beat). `tuned` = base + your LoRA adapter.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE_MODEL  = "facebook/nllb-200-distilled-600M"
ADAPTER_DIR = "models/nllb_bikol_lora"
SRC_LANG, TGT_LANG = "bcl_Latn", "eng_Latn"
MAX_LEN = 128
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
base  = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device).eval()
tuned = PeftModel.from_pretrained(
    AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL), ADAPTER_DIR).to(device).eval()
print("both models loaded on", device)

## 4. Translate the test set with both models
Greedy decoding, batched. ~2–4 min on a T4.

In [ ]:
import torch

def translate_all(model, texts, batch_size=16):
    out = []
    forced = tokenizer.convert_tokens_to_ids(TGT_LANG)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=forced,
                                 max_length=MAX_LEN, num_beams=1)
        out.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return out

src   = test_df[SRC_COL].tolist()
refs  = test_df[TGT_COL].tolist()
print("translating with base model...");  hyp_base  = translate_all(base, src)
print("translating with tuned model..."); hyp_tuned = translate_all(tuned, src)
print("done — example:")
print("  SRC :", src[0]); print("  REF :", refs[0])
print("  BASE:", hyp_base[0]); print("  TUNED:", hyp_tuned[0])

## 5. Compute BLEU, chrF++, TER (criteria D1, D2)
Using SacreBLEU. chrF++ = chrF with word-order component (word_order=2).

In [ ]:
import sacrebleu, json

def score(hyps, refs):
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score     # chrF++
    ter  = sacrebleu.metrics.TER().corpus_score(hyps, [refs]).score
    return round(bleu,2), round(chrf,2), round(ter,2)

b_bleu, b_chrf, b_ter = score(hyp_base,  refs)
t_bleu, t_chrf, t_ter = score(hyp_tuned, refs)

print(f"sacrebleu version: {sacrebleu.__version__}")
print(f"{'Model':<28}{'BLEU':>8}{'chrF++':>8}{'TER':>8}")
print("-"*52)
print(f"{'Zero-shot base (un-tuned)':<28}{b_bleu:>8}{b_chrf:>8}{b_ter:>8}")
print(f"{'LoRA fine-tuned (yours)':<28}{t_bleu:>8}{t_chrf:>8}{t_ter:>8}")
print(f"{'Difference':<28}{round(t_bleu-b_bleu,2):>8}{round(t_chrf-b_chrf,2):>8}{round(t_ter-b_ter,2):>8}")
print("\n(BLEU & chrF++ higher = better; TER lower = better)")

metrics = {
  "test_size": len(refs), "decoding": "greedy (num_beams=1)",
  "sacrebleu_version": sacrebleu.__version__,
  "base":  {"BLEU": b_bleu, "chrF++": b_chrf, "TER": b_ter},
  "tuned": {"BLEU": t_bleu, "chrF++": t_chrf, "TER": t_ter},
}
os.makedirs("results", exist_ok=True)
json.dump(metrics, open("results/metrics.json","w"), indent=2)
print("\nsaved results/metrics.json")

## 6. Error analysis — worst 40 test sentences (criterion D3)

We sort the fine-tuned model's outputs by sentence-level chrF (worst first) and take 40 — the actual
mistranslations. A **suggested category** is auto-filled only for the clear case (an untranslated copy);
the rest are left blank for **you** to categorize using the three rubric categories:
- **Untranslated token (OOV)** — source word copied / left in the output
- **Morphological or prefix error** — wrong affix/inflection on an otherwise right root
- **Word order** — right words, wrong order
(Add an "Other" row only if you explain it.)

In [ ]:
import sacrebleu, pandas as pd

# sentence-level chrF++ for the tuned model
sent_chrf = [sacrebleu.sentence_chrf(h, [r], word_order=2).score
             for h, r in zip(hyp_tuned, refs)]

rows = sorted(zip(src, refs, hyp_tuned, sent_chrf), key=lambda x: x[3])[:40]

def guess(src_t, hyp_t):
    s = set(src_t.lower().split()); h = hyp_t.lower()
    if not hyp_t.strip(): return "Untranslated token (OOV)"
    # if a source content word is copied verbatim into the output
    if any(len(w) > 3 and w in h.split() for w in s): return "Untranslated token (OOV)"
    return ""   # leave blank for manual categorization

err = pd.DataFrame([{
    "No.": i+1, "source_sentence": s, "reference_translation": r,
    "system_output": h, "sentence_chrF++": round(c,1),
    "error_category": guess(s, h), "note": ""
} for i,(s,r,h,c) in enumerate(rows)])

err.to_csv("results/error_analysis.csv", index=False)
print("saved results/error_analysis.csv  (40 rows)")
print("auto-suggested categories:", (err.error_category != "").sum(), "of 40 — fill in the rest yourself")
err.head(10)

### How to finish the error analysis
1. Download `results/error_analysis.csv` (next cell) and open it.
2. Fill the **error_category** for every row using the 3 categories above; add a short **note**.
3. Count how many fall in each category — that gives the numbers for report Section 5.3:
   *Untranslated [n]; Morphological/prefix [n]; Word order [n]; Other [n].*
The filled table becomes **Appendix A** of your report.

## 7. Save & download all Phase 4 results

In [ ]:
import shutil
# bundle metrics + error table for your report / repo
shutil.make_archive("phase4_results", "zip", ".", "results")
from google.colab import files
print("Files in results/:", os.listdir("results"))
files.download("phase4_results.zip")

---
### For the report (Section 5, criteria D1–D4)
- **D1 metrics:** BLEU/chrF++/TER from cell 5 (state SacreBLEU version — printed).
- **D2 baseline comparison:** the base-vs-tuned table + the Difference row.
- **D3 error analysis:** `error_analysis.csv` (40 sentences) → fill categories → Appendix A + counts.
- **D4 interpretation:** explain what the numbers mean — e.g. *what chrF++ shows that BLEU doesn't*
  (chrF++ scores partial/character overlap, so it credits near-misses and morphology that BLEU's
  n-gram matching misses — important for a morphologically rich, low-resource language). State the
  limit: a 283-sentence test set gives wide confidence intervals.